RAG for Materials Science — Your Prototype's Knowledge Layer

5 PDFs → OpenAI embeddings → ChromaDB → retrieved chunks → grounded answer.

1. Load

PyPDFDirectoryLoader reads every PDF in a folder → one Document per page, with source and page already in .metadata. Keep that metadata: it becomes your citations

In [1]:
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

from langchain_community.document_loaders import PyPDFDirectoryLoader

pages = PyPDFDirectoryLoader("papers/").load()
print(pages[0].metadata)

/tmp/ipykernel_19145/1055002081.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-10-03T01:55:50+00:00', 'author': '', 'keywords': '', 'moddate': '2023-10-03T01:55:50+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'papers/chemcrow.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1'}


2. Split

Why split at all? The two failure modes pull in opposite directions:

Chunks too big → the embedding is an average of many ideas, so it matches nothing sharply, and you waste context pasting irrelevant text.
Chunks too small → the fact gets severed from what makes it meaningful. "0.039 eV/atom" is useless without "formation energy MAE".

RecursiveCharacterTextSplitter tries paragraph → line → sentence → word, breaking at the most natural boundary that fits. The overlap is insurance for facts that straddle a cut.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
).split_documents(pages)


In [3]:
len(chunks)

1070

3–4. Embed and store in ChromaDB

An embedding turns text into a vector whose direction encodes meaning. Chunks about formation energy point one way, chunks about sourdough point another. Retrieval = find the vectors pointing the same way as the question.

In [4]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_DIR, DB_DIR, COLLECTION = "papers", "./chroma_db", "materials_papers"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 1. Always open the store. Cheap: no embedding happens here.
vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,   # constructor: embedding_function=
    persist_directory=DB_DIR,
)

# 2. Ask the collection whether it's populated — not whether a folder exists.
if vectorstore._collection.count() == 0:
    print("Empty collection — indexing (this costs money, once)...")

    pages = PyPDFDirectoryLoader(PDF_DIR).load()
    if not pages:
        raise SystemExit(f"No PDFs in ./{PDF_DIR}/")

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=200, add_start_index=True,
    ).split_documents(pages)

    # 3. add_documents() embeds and writes into the store we already opened.
    vectorstore.add_documents(chunks)
    print(f"Indexed {len(chunks)} chunks -> {vectorstore._collection.count()} rows")
else:
    print(f"Reusing existing index: {vectorstore._collection.count()} chunks")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

Empty collection — indexing (this costs money, once)...
Indexed 1070 chunks -> 1070 rows


5a. Retrieve — inspect the chunks before you trust any answer

Query the store on its own first. If the wrong chunks come back, no prompt engineering downstream will save the answer.

In [5]:
for doc, score in vectorstore.similarity_search_with_score(
        "What is the formation energy prediction error?", k=4):
    print(f"{score:.3f}  {doc.metadata['source']} p.{doc.metadata['page']}")
    print(f"       {doc.page_content[:100]}…")

0.835  papers/omat-2024.pdf p.24
       the 222 compounds is 55 meV/atom. Figure C.2 shows the difference in predicted and experiemntal form…
0.850  papers/omat-2024.pdf p.25
       −4 −3 −2 −1 0
−4
−3.5
−3
−2.5
−2
−1.5
−1
−0.5
0
Exp formation energy (eV/atom)
OMat24 formation ener…
0.919  papers/mace-mp-0.pdf p.107
       MACE-MPA-0 cuts this error more than in half, achieving a formation energy MAE of 28 meV/atom.
Simil…
0.941  papers/omat-2024.pdf p.4
       settings. In order to compute formation energies, we computed elemental references and fit anion and…


5b. Generate a grounded answer

Retrieve → format chunks into the prompt → instruct the model to answer only from them.

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the context does not contain the answer, say you don't know. "
    "Cite the source file and page for each claim.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    """Inject source+page INTO the context — the model can only cite what it sees."""
    return "\n\n".join(
        f"[{os.path.basename(d.metadata['source'])} p.{d.metadata['page']}]\n{d.page_content}"
        for d in docs
    )

llm = ChatOpenAI(model="gpt-5.5")

def answer(question: str) -> str:
    docs = retriever.invoke(question)
    if not docs:
        return "No relevant passages found."
    return llm.invoke(prompt.format(context=format_docs(docs), question=question)).content

In [7]:
#test it out

print(answer("What is ChemCrow?"))
print(answer("Is color red a good a color?"))
print(answer("What are the universal model of atoms?"))

ChemCrow is a novel LLM-powered chemistry system that integrates computational tools with the reasoning power of large language models to support chemistry tasks. It is designed to act as an assistant for expert chemists and to provide non-experts with a simple interface for accessing accurate chemical knowledge. It can be adapted to new applications by providing tools and descriptions of their intended use in natural language. (chemcrow.pdf p.2; chemcrow.pdf p.6)
I don’t know. The provided context discusses ChemCrow, chromophore discovery, and chemical reaction tools, but it does not state whether red is a good color. (chemcrow.pdf pp. 2–3, 10)
Universal Models for Atoms (UMA) are a family of models from Meta FAIR designed to quickly and accurately compute properties from atomic simulations, with an emphasis on speed, accuracy, and generalization across chemistry and materials science. They are trained on about half a billion unique 3D atomic structures compiled across domains such as

Wiring it into the agent

Wrap the retriever in @tool and it becomes just another thing the agent chooses among — your private corpus alongside web search.

In [ ]:
from langchain_core.tools import tool

@tool
def search_papers(query: str) -> str:
    """Search the local materials-science paper collection for passages relevant
    to the query. Use this BEFORE web search for questions about materials
    property prediction — these are peer-reviewed papers, not web pages."""
    return format_docs(retriever.invoke(query)) or "No relevant passages found."

#agent = create_agent(model=llm, tools=[search_papers, web_search, calculator], ...)